<a href="https://colab.research.google.com/github/cauavorpagel/pcd-aulas-iniciais/blob/main/Estudo_Independete_III.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q beautifulsoup4 pandas

In [2]:
!unzip -q Filmes_mais_populares_files.zip -d Filmes_mais_populares_files_files

unzip:  cannot find or open Filmes_mais_populares_files.zip, Filmes_mais_populares_files.zip.zip or Filmes_mais_populares_files.zip.ZIP.


In [3]:
import json
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files

In [4]:
# o site do IMDB estava bloqueando o acesso para fazer a raspagem, sendo assim optei por salvar um arquivo .html estático.
caminho_arquivo = "Filmes mais populares.html"

with open(caminho_arquivo, "r", encoding="utf-8") as file:
    html_carregado = file.read()

In [5]:
lista_filmes = []

In [6]:
soup = BeautifulSoup(html_carregado, "html.parser")

script_json = soup.find('script', type='application/ld+json')

In [7]:
if script_json:

  dados = json.loads(script_json.string)

  for item in dados.get('itemListElement', []):
        lista_filme_info = item.get('item', {})

        rating_info = lista_filme_info.get('aggregateRating', {})
        nota = rating_info.get('ratingValue') if rating_info else None

        lista_filmes.append({
            'Nome': lista_filme_info.get('name'),
            'Nome BR': lista_filme_info.get('alternateName', lista_filme_info.get('name')),
            'Nota IMDb': nota,
            'Gênero': lista_filme_info.get('genre'),
            'Duração': lista_filme_info.get('duration'),
            'Classificação': lista_filme_info.get('contentRating'),
            'Poster': lista_filme_info.get('image')
        })

In [8]:
print(f"Total de filmes encontrados: {len(lista_filmes)}")

Total de filmes encontrados: 100


In [9]:
df = pd.DataFrame(lista_filmes)


In [10]:
qtd_visualizar = int(input("Quantos filmes você deseja visualizar? "))

df.head(qtd_visualizar)

Quantos filmes você deseja visualizar? 10


,Nome,Nome BR,Nota IMDb,Gênero,Duração,Classificação,Poster
0,You Can See Everything,Tudo Pode Ser Visto,9.0,"Documentário, Biografia, Comédia",PT2H54M,R,https://m.media-amazon.com/images/M/MV5BOTUxMW...
1,Godzilla Minus Zero,Godzilla Minus Zero,7.9,"Ação, Drama, Terror",PT2H15M,16,https://m.media-amazon.com/images/M/MV5BNTQzZT...
2,Heart of the Beast,Coração Selvagem,7.1,"Ação, Aventura, Drama",PT1H41M,14,https://m.media-amazon.com/images/M/MV5BMDVlMT...
3,Resident Evil,Resident Evil,7.4,Terror,PT1H34M,18,https://m.media-amazon.com/images/M/MV5BNDk4OD...
4,The Odyssey,A Odisseia,8.4,"Ação, Aventura, Drama",PT2H52M,14,https://m.media-amazon.com/images/M/MV5BNTcyNm...
5,Ramayana: Rise of a Legend,Ramayana,8.3,"Ação, Aventura, Drama",PT3H15M,PG-13,https://m.media-amazon.com/images/M/MV5BYWE4NW...
6,Wild Horse Nine,Cavalo Selvagem Nove,8.1,"Ação, Comédia, Policial",PT1H56M,16,https://m.media-amazon.com/images/M/MV5BM2Q3OD...
7,Wildwood,O Bosque Selvagem,8.6,"Animação, Aventura, Fantasia",PT2H10M,14,https://m.media-amazon.com/images/M/MV5BYjk1Y2...
8,Clayface,Cara-de-Barro,7.5,"Terror, Ficção científica, Suspense",PT1H48M,18,https://m.media-amazon.com/images/M/MV5BNTdjZW...
9,Primetime,Primetime,6.9,"Biografia, Policial, Drama",PT1H49M,R,https://m.media-amazon.com/images/M/MV5BMTcyZG...


In [11]:
json = "destaques_semanais.json"

df.to_json(
    json,
    orient="records",
    force_ascii=False,
    indent=4
)

files.download(json)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from IPython.display import display, Image

busca_filme = input("Digite o nome do filme: ").strip().lower()

nome_lower = df['Nome'].fillna('').str.lower()
nome_br_lower = df['Nome BR'].fillna('').str.lower()

resultado = df[
    nome_lower.str.contains(busca_filme) |
    nome_br_lower.str.contains(busca_filme)
]

if not resultado.empty:
    # Pega o primeiro filme encontrado
    filme = resultado.iloc[0]

    print(f"🎬 {filme['Nome BR']} ({filme['Nome']})")
    print(f"⭐ Nota IMDb: {filme['Nota IMDb']} | 🎭 Gênero: {filme['Gênero']}")
    print(f"⏱️ Duração: {filme['Duração']} | 🔞 Classificação: {filme['Classificação']}\n")

    # Exibe o pôster direto da URL com largura ajustada (ex: 250px)
    if filme['Poster']:
        display(Image(url=filme['Poster'], width=250))
else:
    print("Nenhum filme encontrado.")

Digite o nome do filme: Obsession
🎬 Obsessão (Obsession)
⭐ Nota IMDb: 7.8 | 🎭 Gênero: Fantasia, Terror, Suspense
⏱️ Duração: PT1H49M | 🔞 Classificação: 18

